# 🧠 Introduction to Neural Networks
### A Step-by-Step Beginner Notebook
---
**What you will learn:**
- What a neural network is (with a real analogy)
- How neurons, weights, and activations work
- How the network learns (forward pass → loss → backprop → update)
- Build and train your first neural network using Keras
- Plot training loss and accuracy
- Make predictions on new data

> **Dataset:** Breast Cancer Wisconsin (built into scikit-learn — no download needed!)

---
## 📦 Step 1 — Install and Import Libraries

We need:
- `numpy` and `pandas` — for data handling
- `matplotlib` — for plotting
- `sklearn` — for dataset and preprocessing
- `tensorflow / keras` — to build and train the neural network

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras

print('TensorFlow version:', tf.__version__)
print('All libraries imported successfully!')

---
## 📊 Step 2 — Load and Explore the Dataset

We will use the **Breast Cancer Wisconsin** dataset.

- **Task:** Predict if a tumor is **Malignant (cancerous)** or **Benign (not cancerous)**
- **Features:** 30 measurements from cell images (radius, texture, area, etc.)
- **Target:** 0 = Malignant,  1 = Benign

In [ ]:
# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target

print('Dataset shape (rows x columns):', X.shape)
print('Number of features:', X.shape[1])
print('Number of samples:', X.shape[0])
print()
print('Feature names:', data.feature_names[:5], '...')
print()
print('Target classes:', data.target_names)
print('Class distribution — Malignant (0):', np.sum(y==0), '  Benign (1):', np.sum(y==1))

In [ ]:
# Look at first 5 rows as a table
df = pd.DataFrame(X, columns=data.feature_names)
df['target'] = y
df.head()

---
## ✂️ Step 3 — Split Data into Train and Test

We split the data:
- **80%** for training — the network learns from this
- **20%** for testing — we check how well it learned on data it has NEVER seen

Think of it like: study from a textbook, then take an exam on new questions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training samples:', X_train.shape[0])
print('Testing samples :', X_test.shape[0])

---
## ⚖️ Step 4 — Normalize the Data

Neural networks are very sensitive to the **scale** of features.

Example problem without normalization:
- `mean_radius = 14.0`
- `mean_smoothness = 0.09`

The radius dominates just because its number is bigger — not because it is more important.

**StandardScaler** transforms each feature to have mean=0 and std=1.

> ⚠️ **Important:** Fit the scaler on TRAIN data only, then apply to both train and test.

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)  # Fit on train, transform train
X_test  = scaler.transform(X_test)       # Only transform test (no fitting!)

print('Before normalization — original first sample, first 3 features:')
print('  ', data.data[0, :3])
print('After normalization — first 3 features:')
print('  ', X_train[0, :3].round(3))

---
## 🏗️ Step 5 — Build the Neural Network

We will build a simple network with:
- **Input layer:** 30 neurons (one per feature)
- **Hidden layer 1:** 16 neurons, ReLU activation
- **Hidden layer 2:** 8 neurons, ReLU activation
- **Output layer:** 1 neuron, Sigmoid activation (gives probability 0 to 1)

```
Input (30)  -->  Hidden 1 (16, ReLU)  -->  Hidden 2 (8, ReLU)  -->  Output (1, Sigmoid)
```

In [ ]:
# Set random seed so results are reproducible
tf.random.set_seed(42)

model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(30,)),  # Hidden layer 1
    keras.layers.Dense(8,  activation='relu'),                     # Hidden layer 2
    keras.layers.Dense(1,  activation='sigmoid')                   # Output layer
])

model.summary()

### 🔍 Understanding model.summary()

| Column | Meaning |
|---|---|
| Layer (type) | Name and type of the layer |
| Output Shape | Shape of data coming out of this layer |
| Param # | Number of weights + biases in this layer |

---
## ⚙️ Step 6 — Compile the Model

Before training we must **compile** the model — telling it:
- **Optimizer:** How to update weights (`adam` is the best default choice)
- **Loss function:** How to measure mistakes (`binary_crossentropy` for Yes/No problems)
- **Metric:** What to track (`accuracy` — percentage of correct predictions)

In [ ]:
model.compile(
    optimizer = 'adam',
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)

print('Model compiled and ready to train!')

---
## 🚀 Step 7 — Train the Model

Now the real learning happens!

- `epochs=100` — the network will go through all training data 100 times
- `batch_size=32` — update weights after every 32 samples
- `validation_split=0.2` — use 20% of training data to monitor overfitting

Watch the loss go **down** and accuracy go **up** — that means the model is learning!

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs          = 100,
    batch_size      = 32,
    validation_split= 0.2,
    verbose         = 1
)

---
## 📈 Step 8 — Plot Training History

We plot two things:
1. **Loss over epochs** — should go down for both train and validation
2. **Accuracy over epochs** — should go up for both

If training accuracy is much higher than validation accuracy → **overfitting** (model memorized data)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Loss
axes[0].plot(history.history['loss'],     label='Train Loss',      color='blue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='red', linestyle='--')
axes[0].set_title('Loss over Epochs', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Plot Accuracy
axes[1].plot(history.history['accuracy'],     label='Train Accuracy',      color='blue')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', color='red', linestyle='--')
axes[1].set_title('Accuracy over Epochs', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## 🧪 Step 9 — Evaluate on Test Data

Now we test the model on data it has **NEVER seen** during training.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_accuracy * 100:.2f}%')

---
## 🔮 Step 10 — Make Predictions

In [ ]:
# Get probability predictions
y_prob = model.predict(X_test).flatten()

# Convert probabilities to class labels (threshold = 0.5)
y_pred = (y_prob >= 0.5).astype(int)

# Show first 10 predictions
print('Sample Predictions:')
print(f'{"Sample":>8}  {"True Label":>12}  {"Probability":>13}  {"Predicted":>10}  {"Correct?":>8}')
print('-' * 60)
for i in range(10):
    label_true = data.target_names[y_test[i]]
    label_pred = data.target_names[y_pred[i]]
    correct = '✓' if y_test[i] == y_pred[i] else '✗'
    print(f'{i+1:>8}  {label_true:>12}  {y_prob[i]:>12.3f}  {label_pred:>10}  {correct:>8}')

---
## 🗂️ Step 11 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred Malignant', 'Pred Benign'], fontsize=11)
ax.set_yticklabels(['True Malignant', 'True Benign'], fontsize=11)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=18,
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im)
plt.tight_layout()
plt.show()

print('\nDetailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

---
## 📊 Step 12 — Visualize What the Network Learned
### Plot: Predicted Probability Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(y_prob[y_test == 0], bins=30, alpha=0.6, color='red',  label='True Malignant')
ax.hist(y_prob[y_test == 1], bins=30, alpha=0.6, color='blue', label='True Benign')
ax.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Decision threshold (0.5)')
ax.set_xlabel('Predicted Probability (Benign)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Predicted Probability Distribution by True Class', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Ideal result: red bars should cluster near 0, blue bars should cluster near 1.')
print('This shows the network has learned to separate the two classes well.')

---
## 🎯 Step 13 — Student Exercises

Try these on your own to deepen your understanding:

**Exercise 1:** Change the number of neurons in Hidden Layer 1 from 16 to 32. Does accuracy improve?

**Exercise 2:** Add a third hidden layer with 4 neurons. What happens to training time and accuracy?

**Exercise 3:** Change epochs from 100 to 200. Does the model keep improving or does it overfit?

**Exercise 4:** Change the activation in hidden layers from `relu` to `tanh`. Compare the results.

**Exercise 5:** Change `batch_size` from 32 to 8. Does training become faster or slower?

In [ ]:
# ── Exercise workspace — write your code here ─────────────────────────────────

model2 = keras.Sequential([
    keras.layers.Dense(___,  activation=___,  input_shape=(30,)),
    keras.layers.Dense(___,  activation=___),
    keras.layers.Dense(1,    activation='sigmoid')
])

model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history2 = model2.fit(X_train, y_train, epochs=___, batch_size=___, validation_split=0.2, verbose=1)

loss2, acc2 = model2.evaluate(X_test, y_test, verbose=0)
print(f'Model 2 Test Accuracy: {acc2 * 100:.2f}%')

---
## 📝 Summary — What You Learned

| Concept | What it is |
|---|---|
| Neuron | A unit that takes inputs, multiplies by weights, adds bias, applies activation |
| Weight | A number saying how important each input is — learned during training |
| Activation | A function (ReLU, Sigmoid) that adds non-linearity to the neuron's output |
| Forward Pass | Data flows through the network to produce a prediction |
| Loss | A number measuring how wrong the prediction is |
| Backpropagation | Going backwards to find which weights caused the error |
| Gradient Descent | Updating weights step by step to reduce the loss |
| Epoch | One full pass through all training data |
| Overfitting | Model memorizes training data but fails on new data |
